# main_pipeline (Kaggle)

Orchestrator chạy toàn bộ pipeline (build data -> train -> chat demo -> evaluate)
trên Kaggle (có GPU free).

In [ ]:
pip install -q --force-reinstall --no-cache-dir "numpy==1.26.4" "pandas==2.2.2" "pyarrow>=14,<18"

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["HF_HUB_DISABLE_XET"] = "1"

In [2]:
!git clone --branch tuanphat --single-branch https://github.com/internvietridao/MockProject_062026_NhomAI.git

Cloning into 'MockProject_062026_NhomAI'...
remote: Enumerating objects: 11550, done.
remote: Counting objects: 100% (41/41), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 11550 (delta 23), reused 20 (delta 16), pack-reused 11509 (from 3)
Receiving objects: 100% (11550/11550), 172.84 MiB | 32.50 MiB/s, done.
Resolving deltas: 100% (6896/6896), done.
Updating files: 100% (11418/11418), done.


In [3]:
PROJECT_DIR = "/kaggle/working/MockProject_062026_NhomAI/Training"

import sys
sys.path.insert(0, PROJECT_DIR)

%cd /kaggle/working/MockProject_062026_NhomAI/Training

/kaggle/working/MockProject_062026_NhomAI/Training


In [4]:
!pip install -q -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 85.7 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.9/503.9 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.9/511.9 kB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.9/374.9 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 30.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.8/494.8 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.7/175.7 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 90.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.4/396.4 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━

In [5]:
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
print("ragas import OK")

ragas import OK


## Bước 1: Build train.jsonl từ final_train_dataset.json

In [6]:
from pipeline import build_train_dataset
build_train_dataset.main()

Total samples (raw)   : 37172
Sample limit áp dụng  : None
Hợp lệ (validated)    : 37172
Bỏ qua (thiếu Q/A)    : 0
USE_RAG               : False
----------------------------------------
Train : 26020 -> /kaggle/working/MockProject_062026_NhomAI/Training/output/train.jsonl
Val   : 5575 -> /kaggle/working/MockProject_062026_NhomAI/Training/output/val.jsonl
Test  : 5577 -> /kaggle/working/MockProject_062026_NhomAI/Training/output/test.jsonl


## Bước 2: Train LoRA

In [ ]:
from pipeline import train
train.main()

### Dùng khi đã có output model nếu không muốn train lại

In [7]:
from pipeline import run_evaluation
run_evaluation.main()                    # tự động full test set
#run_evaluation.main(resume=True)         # full, nối tiếp CSV cũ

Đang tải tập test (thô)...
Đang load model đã train từ /kaggle/working/MockProject_062026_NhomAI/Training/output/output_model...


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Load xong.
Số câu hỏi test: 5577 | Sẽ chạy: 5577
[EVAL] Bắt đầu MỚI -- sẽ ghi đè /kaggle/working/MockProject_062026_NhomAI/Training/output/evaluation_results.csv nếu đã tồn tại.
[EVAL] Mục tiêu: 5577 mẫu.
[EVAL] Đã inference 5/5577 mẫu... (đã lưu CSV)
[EVAL] Đã inference 10/5577 mẫu... (đã lưu CSV)
[EVAL] Đã inference 15/5577 mẫu... (đã lưu CSV)
[EVAL] Đã inference 20/5577 mẫu... (đã lưu CSV)
[EVAL] Đã inference 25/5577 mẫu... (đã lưu CSV)
[EVAL] Đã inference 30/5577 mẫu... (đã lưu CSV)
[EVAL] Đã inference 35/5577 mẫu... (đã lưu CSV)
[EVAL] Đã inference 40/5577 mẫu... (đã lưu CSV)
[EVAL] Đã inference 45/5577 mẫu... (đã lưu CSV)
[EVAL] Đã inference 50/5577 mẫu... (đã lưu CSV)
[EVAL] Đã inference 55/5577 mẫu... (đã lưu CSV)
[EVAL] Đã inference 60/5577 mẫu... (đã lưu CSV)
[EVAL] Đã inference 65/5577 mẫu... (đã lưu CSV)
[EVAL] Đã inference 70/5577 mẫu... (đã lưu CSV)
[EVAL] Đã inference 75/5577 mẫu... (đã lưu CSV)
[EVAL] Đã inference 80/5577 mẫu... (đã lưu CSV)
[EVAL] Đã inference 85/5577 

KeyboardInterrupt: 

## Bước 3: Demo chat (inference)

In [ ]:
from pipeline import chat
chat.main()

## (Tuỳ chọn) Lưu output_model về lại Kaggle Output

`output/output_model/` đã nằm trong `/kaggle/working/`, Kaggle sẽ tự lưu làm
Output của notebook này khi Save Version — không cần thêm gì.

In [ ]:
%cd /kaggle/working/MockProject_062026_NhomAI/Training
!zip -r output.zip output